# Demand Forecasting & Inventory Optimization
## Section 4: Inventory Logic

**Goal of this section:** Turn our forecast numbers and their errors into actual business decisions—specifically, **how much stock to hold (Safety Stock)** and **when to reorder (Reorder Point)**. This is where our project transitions from an analysis to an automated decision-making tool.

We will focus on our Top 5 "Class A" products. For each, we will calculate:
1. **Demand Variability:** Standard deviation of forecast errors (what causes stockouts).
2. **Safety Stock:** The buffer inventory we need to keep a 95% service level.
3. **Reorder Point:** The specific inventory level that triggers a new order.
4. **Economic Order Quantity (EOQ):** The optimal amount to order to balance ordering costs and holding costs.
5. **Naive Comparison:** How this compares to a naive "1.5x last month's demand" policy.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

# Load the clean dataset
df = pd.read_csv('clean_supply_chain_data.csv', parse_dates=['order_date'])
print("Data loaded successfully!")

Data loaded successfully!


### Step 1: Set Up Assumptions & Run Inventory Logic
We'll iterate through our Top 5 products. For each product, we will re-calculate the Holt-Winters forecast error to get our **demand variability**. 

**Key Assumptions:**
* **Lead Time:** We will use the average `actual_lead_time` from the dataset for each product.
* **Service Level:** 95% (We are okay with running out of stock only 5% of the time).
* **Ordering Cost:** $500 per order placed.
* **Holding Cost:** $20 per unit annually.

In [2]:
# Top 5 Class A Products
top_products = [
    "Field & Stream Sportsman 16 Gun Fire Safe",
    "Perfect Fitness Perfect Rip Deck",
    "Diamondback Women's Serene Classic Comfort Bi",
    "Nike Men's Free 5.0+ Running Shoe",
    "Nike Men's Dri-FIT Victory Golf Polo"
]

inventory_data = []

ordering_cost = 500
holding_cost_per_unit = 20
service_level = 0.95
z_score = st.norm.ppf(service_level) # ≈ 1.645

for product in top_products:
    # 1. Prepare Time Series
    prod_df = df[df['product_name'] == product]
    ts = prod_df.groupby(prod_df['order_date'].dt.to_period('M'))['quantity_sold'].sum()
    ts.index = ts.index.to_timestamp()
    ts = ts.asfreq('MS', fill_value=0)
    
    # Lead time & Unit Price
    lead_time_days = prod_df['actual_lead_time'].mean()
    unit_price = prod_df['unit_price'].mean()
    
    # 2. Train/Test Split (Hold out last 6 months)
    train = ts.iloc[:-6]
    test = ts.iloc[-6:]
    
    # 3. Fit Holt-Winters & get Forecast Error
    # We use additive trend and seasonality (12 months)
    hw_model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
    hw_forecast = hw_model.forecast(len(test))
    
    forecast_errors = test.values - hw_forecast.values
    demand_std = np.std(forecast_errors)
    
    # 4. Calculate Safety Stock
    # Safety Stock = Z * std_dev_of_error * sqrt(lead_time_days/30) (converting lead time to months since error is monthly)
    # Actually, let's keep it simple: since demand_std is monthly, let's convert lead time to months.
    lead_time_months = lead_time_days / 30.0
    safety_stock = z_score * demand_std * np.sqrt(lead_time_months)
    
    # Ensure safety stock is not negative
    safety_stock = max(0, safety_stock)
    
    # 5. Calculate Reorder Point
    avg_daily_demand = train.mean() / 30.0
    reorder_point = (avg_daily_demand * lead_time_days) + safety_stock
    
    # 6. Calculate EOQ
    annual_demand = train.sum() * (12 / len(train))
    eoq = np.sqrt((2 * annual_demand * ordering_cost) / holding_cost_per_unit)
    
    # 7. Naive Comparison (1.5x last month of train data)
    naive_stock_level = train.iloc[-1] * 1.5
    optimized_stock_level = reorder_point + (eoq / 2)
    
    inventory_data.append({
        'Product': product,
        'Lead_Time_Days': round(lead_time_days, 1),
        'Demand_Std_Error': round(demand_std, 1),
        'Safety_Stock': int(round(safety_stock)),
        'Reorder_Point': int(round(reorder_point)),
        'EOQ': int(round(eoq)),
        'Naive_Stock_Level': int(round(naive_stock_level)),
        'Optimized_Avg_Stock': int(round(optimized_stock_level))
    })

inventory_table = pd.DataFrame(inventory_data)
inventory_table

,Product,Lead_Time_Days,Demand_Std_Error,Safety_Stock,Reorder_Point,EOQ,Naive_Stock_Level,Optimized_Avg_Stock
0,Field & Stream Sportsman 16 Gun Fire Safe,3.5,196.2,110,171,559,776,451
1,Perfect Fitness Perfect Rip Deck,3.5,754.9,425,686,1159,3336,1266
2,Diamondback Women's Serene Classic Comfort Bi,3.5,158.3,89,137,498,654,386
3,Nike Men's Free 5.0+ Running Shoe,3.5,391.4,220,349,817,1629,758
4,Nike Men's Dri-FIT Victory Golf Polo,3.5,706.6,396,618,1069,2553,1152


### Step 2: Save the Inventory Policy
Let's save this optimized inventory policy table to a CSV. This will be the direct feed for our final dashboard in Section 6.

In [3]:
inventory_table.to_csv('inventory_policy.csv', index=False)
print("Inventory policy saved to 'inventory_policy.csv'")

Inventory policy saved to 'inventory_policy.csv'


###  Section 4 Summary

* **The Problem:** We knew our forecasts were accurate, but a forecast alone doesn't tell the warehouse manager what to do. They need exact numbers: "How many extra units do I keep as a buffer?" and "At what stock level do I click 'Buy' again?"
* **The Result:** We calculated the standard deviation of our forecast *errors* to measure true demand variability. Using a 95% service level, we determined the exact **Safety Stock** and **Reorder Point** for our Top 5 products. We also calculated the **Economic Order Quantity (EOQ)** to balance ordering and holding costs. Finally, we compared our optimized average stock level to a naive policy (just stocking 1.5x last month's demand), giving us the exact numbers needed to prove the financial value of our model in the next section.